# 用自然语言控制 MicroDuck
官方 MicroDuck 提供的是**身体**：行走、起身、踢球这些动作，接收的是速度、姿态等低层指令（见[官方展示](https://colab.research.google.com/github/atlasmarkz/microduck-qwen/blob/main/notebooks/microduck-baseline.ipynb)）。

本仓库加上的是**听懂一句话并把它做完的能力**：

> 你说：“去别的房间找蓝色的球，走到它旁边，然后回起点”
>
> 机器人：`探索找蓝球` → `走近蓝球` → `返回起点` → `任务完成`

1. **听和看**：Qwen3-VL-4B 根据你的话、相机图像和已完成的动作，选择下一个技能。
2. **走**：执行器用深度图建图、规划路径，由官方行走步态完成移动，净空不足就停车。
3. **再看一眼**：每做完一步重新取图再决定，直到模型自己判断任务结束；目标不明确时会反问你。

## 先看效果：同一句话，四种控制方式

不用运行，也不用 GPU。四个面板是**同一个任务、同一个布局**的仿真实录，从左上到右下逐层加上本仓库的方法。每个面板左边是俯视轨迹，右边是**跟随机器人的第三视角**（只给观众看，模型看不到），下面是模型的决策和原始输出。这批录像只记录了机器人的位置和朝向，腿部画为默认站姿；带完整关节姿态的补跑完成后会替换。

| 控制方式 | 成功 | 失败 | 评测截断 |
|---|---:|---:|---:|
| ① 裸 Qwen：只有基本动作 | **0/80** | 80 | 0 |
| ② 加技能框架 | **27/80** | 4 | 49 |
| ③ 再加结束规则 | **68/80** | 1 | 11 |
| ④ 再加 LoRA（完整系统） | **74/80** | 3 | 3 |

80 个任务来自 16 个布局，模型训练和框架开发都没有用过（③ 的结束规则例外，见文末）。成败由仿真真值判定：是否到达目标、是否返回、有无碰撞、是否自己结束任务。

> **关于“评测截断”**：②③④ 用的评测会在模型第一次偏离参考动作时停止，所以这些题看不到自然结局，也不算成功。② 的 27 题因此是下限，不是它自由运行的成绩。不截断的补跑完成后会替换这里的数字和录像。

**靠近照片里最左侧的球，之后返回起点**　（走近并返回；80 题中 39 题是这种结果组合）

![靠近照片里最左侧的球，之后返回起点](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-1.webp)

**往前移动20厘米**　（定距移动；80 题中 39 题是这种结果组合）

![往前移动20厘米](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-2.webp)

**在别处找到红球并靠近它，之后返回出发点**　（跨房间找球；80 题中 39 题是这种结果组合）

![在别处找到红球并靠近它，之后返回出发点](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-3.webp)

**拜托你旁边房间里有那个绿色的球儿，过去，到了停下吧**　（跨房间找球；80 题中 22 题是这种结果组合）

![拜托你旁边房间里有那个绿色的球儿，过去，到了停下吧](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-4.webp)

**旁边房间里有那个红色的球儿，过去！**　（跨房间找球；80 题中 7 题是这种结果组合）

![旁边房间里有那个红色的球儿，过去！](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-5.webp)

**靠近照片里最左侧的球，之后返回起点**　（走近并返回；80 题中 3 题是这种结果组合）

![靠近照片里最左侧的球，之后返回起点](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-6.webp)

**please the green ball is in another room, go next to it, and hold still after that**　（跨房间找球；80 题中 2 题是这种结果组合）

![please the green ball is in another room, go next to it, and hold still after that](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-7.webp)

**旁边房间里有那个绿色的球儿，过去吧**　（跨房间找球；80 题中 1 题是这种结果组合）

![旁边房间里有那个绿色的球儿，过去吧](https://raw.githubusercontent.com/atlasmarkz/microduck-qwen/main/assets/replay/task-8.webp)

<small>选题规则：出现不少于 2 次的每种结果组合各取编号最小的一题；再从最常见的组合里补上其他任务类型；最后补完整系统没有成功的题。没有手工挑选。录像里推理等待压缩为固定停顿，运动按同一倍数加速。</small>

<small>适用范围：仿真、单一 4B 模型、红绿蓝黄四色球的 2×2 房间。③ 的结束规则是看过同一批种子的失败后写的，它和 ④ 的差别在这 80 题上不显著（p=0.18，截断口径）。</small>

## 亲自用中文指挥
在 **运行时 → 更改运行时类型** 选择 **GPU（T4 或 L4）**，按顺序运行第 1–4 格。首次运行要下载约 9 GB 基座模型，安装和下载共约 5 分钟；适配器与步态从公开 Hugging Face 仓库免登录拉取。

第 4 格的控制台可以连续输入多句话，也可以点快捷指令。试试换个说法，或者说得含糊一点看它会不会反问。

In [ ]:
# @title 1 · 取出项目源码并安装依赖
project_source = "github"  # @param ["github", "drive", "upload"]
drive_folder = "microduck-qwen"  # @param {type:"string"}
render_backend = "egl"  # @param ["egl", "osmesa"]

from importlib import metadata
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
import zipfile
from pathlib import Path

if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("请选择 Python 3.12 或 3.13 的 GPU 运行时，然后重新运行。")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("请启用 Colab GPU（T4 或 L4），然后重新运行。")
print("GPU:", gpu.stdout.strip())

root = Path("/content/microduck-qwen")
root.mkdir(parents=True, exist_ok=True)
bundle = "microduck-qwen-colab-source.zip"
local_zip = Path("/content") / bundle

if project_source == "github":
    with tempfile.TemporaryDirectory() as update:
        staged = Path(update) / "source"
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/atlasmarkz/microduck-qwen.git", str(staged)], check=True)
        subprocess.run(["git", "-C", str(staged), "sparse-checkout", "set", "microduck_qwen"], check=True)
        actual = subprocess.check_output(["git", "-C", str(staged), "rev-parse", "--short", "HEAD"], text=True).strip()
        shutil.copytree(staged / "microduck_qwen", root / "microduck_qwen", dirs_exist_ok=True)
        shutil.copyfile(staged / "pyproject.toml", root / "pyproject.toml")
        print("已从 GitHub 同步源码，提交：", actual)
elif project_source == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive") / drive_folder
    drive_zip = drive_dir / bundle
    if drive_zip.is_file():
        shutil.copyfile(drive_zip, local_zip)
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(root)
        print("已从 Google Drive 解压最新源码包：", drive_zip)
    elif (drive_dir / "pyproject.toml").is_file() and (drive_dir / "microduck_qwen").is_dir():
        shutil.copytree(drive_dir / "microduck_qwen", root / "microduck_qwen", dirs_exist_ok=True)
        shutil.copyfile(drive_dir / "pyproject.toml", root / "pyproject.toml")
        print("已从 Google Drive 项目目录同步源码：", drive_dir)
    else:
        raise FileNotFoundError(f"未在 {drive_dir} 找到 {bundle} 或项目源码，也可切换到 github 或 upload。")
else:
    from google.colab import files
    uploaded = files.upload()
    if bundle not in uploaded:
        raise FileNotFoundError(f"请选择 {bundle}。")
    local_zip.write_bytes(uploaded[bundle])
    with zipfile.ZipFile(local_zip) as archive:
        archive.extractall(root)

os.environ["MUJOCO_GL"] = render_backend
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _mod in [m for m in list(sys.modules) if m == "microduck_qwen" or m.startswith("microduck_qwen.")]:
    del sys.modules[_mod]

def run(*cmd):
    completed = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        raise RuntimeError(completed.stdout[-8500:])

deps_marker = Path(f"/content/.microduck_deps_gpu_{render_backend}")
if deps_marker.is_file():
    print("当前会话依赖已安装，跳过重复安装。")
else:
    gl_packages = ("libegl1", "libgl1", "fonts-wqy-zenhei") if render_backend == "egl" else ("libosmesa6", "libgl1", "fonts-wqy-zenhei")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    run("apt-get", "-qq", "-y", "install", *gl_packages)
    project = tomllib.loads((root / "pyproject.toml").read_text())["project"]
    requirements = project["optional-dependencies"]
    bam = Path("/content/microduck-bam.zip")
    bam_url = next(item.split(" @ ", 1)[1] for item in requirements["simulation"] if " @ " in item)
    run("curl", "-fsSL", "--retry", "3", bam_url, "-o", str(bam))
    sim_packages = [item for item in requirements["simulation"] if " @ " not in item]
    keep_preinstalled = set()
    try:
        torch_ver = metadata.version("torch")
        print("保留 Colab 预装 PyTorch:", torch_ver, flush=True)
        keep_preinstalled.update({"torch", "torchvision"})
    except metadata.PackageNotFoundError:
        pass
    vision_packages = [
        p for p in requirements["vision-torch"]
        if not p.startswith("Pillow==") and p.split("==")[0].split(">=")[0].strip() not in keep_preinstalled
    ]
    with tempfile.TemporaryDirectory() as build:
        bam_source = bam
        if sys.version_info[:2] == (3, 13):
            with zipfile.ZipFile(bam) as archive:
                archive.extractall(build)
            manifest, = Path(build).glob("bam-*/pyproject.toml")
            config = manifest.read_text(encoding="utf-8")
            guard = 'requires-python = ">=3.12,<3.13"'
            if config.count(guard) != 1:
                raise RuntimeError("固定版 BAM 的 Python 声明与预期不符。")
            manifest.write_text(config.replace(guard, 'requires-python = ">=3.12,<3.14"'),
                                encoding="utf-8")
            bam_source = manifest.parent
        run(sys.executable, "-m", "pip", "install", "-q", "Pillow>=10.1", "ipywidgets",
            *project["dependencies"], *vision_packages, *sim_packages, str(bam_source))
    deps_marker.write_text("ok")
from microduck_qwen.colab_demo import prepare_rendering
print("渲染器：", prepare_rendering(render_backend))
print("依赖安装完成。")

In [ ]:
# @title 2 · 下载固定版本的模型和仿真资产
import tarfile
import tempfile
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download
from microduck_qwen.sim_native import DEFAULT_POLICY, DEFAULT_RL, POLICY_REVISION
from microduck_qwen.vision_torch import MODEL, MODEL_PATH, REVISION, checked_adapter

root = Path("/content/microduck-qwen")
adapter_repo = "atlasmarkz/microduck-skill-agent"
adapter_rev = "f5924d1576b6e5b852320994b0e1db911becd5e6"
assets_dir = root / ".cache" / "colab-adapters"
snapshot_download(adapter_repo, revision=adapter_rev, local_dir=assets_dir,
                  allow_patterns=["lang-v2/*"], token=False)
adapter = assets_dir / "lang-v2"
adapter_report = checked_adapter(adapter, "agent")
print("lang-v2 SHA-256:", adapter_report["adapter_sha256"])

hf_hub_download("pollen-robotics/microduck-policies", "alpha_walking.onnx",
                revision=POLICY_REVISION, cache_dir=DEFAULT_POLICY.parents[3], token=False)

upstream_rev = "0daab160d3fb1c2cbaba16a4a19052f36709dd9e"
if not DEFAULT_RL.exists():
    DEFAULT_RL.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=DEFAULT_RL.parent) as temporary:
        upstream_archive = Path(temporary) / "upstream.tar.gz"
        run("curl", "-fsSL", "--retry", "3",
            f"https://codeload.github.com/pollen-robotics/microduck_rl/tar.gz/{upstream_rev}",
            "-o", str(upstream_archive))
        with tarfile.open(upstream_archive) as archive:
            archive.extractall(temporary, filter="data")
        (Path(temporary) / f"microduck_rl-{upstream_rev}").rename(DEFAULT_RL)

snapshot_download(MODEL, revision=REVISION, cache_dir=MODEL_PATH.parents[2],
                  allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "chat_template.jinja"],
                  token=False)
assert DEFAULT_POLICY.is_file() and MODEL_PATH.is_dir(), "仿真策略或基座模型下载不完整。"
print("模型和仿真资产就绪。")

In [ ]:
# @title 3 · 预热模型，检查真实 MuJoCo 图像和推理
from IPython.display import display
from PIL import Image
from pathlib import Path
from tempfile import TemporaryDirectory

from microduck_qwen.agent import memory_state
from microduck_qwen.colab_demo import DEMO_INSTRUCTION, DEMO_SEED, simulation
from microduck_qwen.navigation import Observer
from microduck_qwen.vision import capture, propose_targets, save_frame
from microduck_qwen.vision_torch import TorchAgentPlanner

if "planner" not in globals():
    planner = TorchAgentPlanner(adapter)
with simulation(DEMO_SEED) as info, TemporaryDirectory() as scratch:
    observer = Observer(info["port"])
    try:
        state = observer.read()
    finally:
        observer.close()
    frame = capture(info["port"], wait_for_fresh=1.5)
    image_path = save_frame(Path(scratch), frame, propose_targets(frame), "warmup")
    preview = planner.predict(DEMO_INSTRUCTION, image_path, memory_state(info, state, []))
    if preview.get("action") is None:
        raise RuntimeError(f"模型预热未生成可执行动作：{preview.get('error', preview.get('raw'))}")
    display(Image.open(image_path).copy())
    print("预热完成，首个动作：", preview["action"])

In [ ]:
# @title 4 · 用中文指挥机器人
seed = 98107  # @param {type:"integer"}

from microduck_qwen.colab_demo import interactive_demo
console = interactive_demo(planner, seed=seed)

In [ ]:
# @title 5 ·（可选）把一次任务录成第三视角 GIF
instruction = "去别的房间找蓝色的球，走到它旁边，然后回起点"  # @param {type:"string"}
seed = 98107  # @param {type:"integer"}
speed = 5  # @param {type:"integer"}
download_to_browser = False  # @param {type:"boolean"}

from microduck_qwen.colab_demo import record_colab_gif
recorded = record_colab_gif(planner, instruction=instruction, seed=seed, speed=speed, drive_folder=drive_folder)
if download_to_browser:
    from google.colab import files
    files.download(recorded["gif"]["path"])

**使用提示**
- 第 1 格应打印 `Tesla T4` 等显卡名；显示 `llvmpipe` 说明退回了软件渲染。
- 日志会逐步显示模型的决策和原始输出。机器人停下来询问时，界面会出现橙色询问框，可以补充说明、重试或取消。
- 随时可点红色 **紧急停止**；**重置场景** 可以换一个 Seed（布局）。
- 控制台是实时界面，不会随 ipynb 保存。第 5 格可把一次任务录成第三视角 GIF；挂载 Drive 后会同步到 `MyDrive/<drive_folder>/`。
- 默认从 GitHub 拉取源码；第 1 格 `project_source` 可改为 `drive` 或 `upload`。

## 实验：踢球和带球
走到球旁以后，再踢一脚或把球带回起点。这是单独的实验格，与上面的演示互不影响；需要先运行第 1–3 格。

- **Qwen 负责**：听懂去哪找哪个球，看图找到并走到球旁。
- **程序负责**：句子里的“踢 / 带球”由规则识别（适配器没有训练过这两个技能），再做对位、踢球或边走边推，最后返回。
- **官方策略**：走路和推球用 `alpha_walking`，踢球用 `ball_kick_left / right`。官方踢球策略看不到球，这里由我们替它对准。
- **与真实情况的差别**：只有指定颜色的球能滚动，质量 15 g（与官方踢球的球相同），半径仍是 7 cm（官方为 3.5 cm），并加了滚动阻力；对位和推球时球的位置直接读自仿真，不是相机看到的。
- **试跑结果**（6 个没用过的布局，各跑一次，不经 Qwen）：踢球 6/6，踢出 0.7–1.0 m，全部返回起点；带球 6/6 把球送到起点 0.5 m 内，其中 1 次球正好停在起点上，机器人回不去。6 个布局里只有 1 个是真正的远距离带球（5.4 m），其余约 1 m，所以还不能说跨房间带球很可靠。[记录](https://github.com/atlasmarkz/microduck-qwen/blob/main/experiments/ball-play-v1/summary.md)

In [ ]:
# @title 6 ·（实验）找到球以后：踢一脚，或把球带回起点
find = "去别的房间找蓝色的球，走到它旁边"  # @param {type:"string"}
then = "把球带回起点"  # @param ["把球带回起点", "踢一脚，然后回起点"] {allow-input: true}
seed = 98107  # @param {type:"integer"}

from microduck_qwen.colab_ball import ball_demo
ball_report = ball_demo(planner, f"{find}，{then}", seed)